# LeetCode 676 — Implement Magic Dictionary bằng Python

Link đề: https://leetcode.com/problems/implement-magic-dictionary/ · Độ khó: **Medium** · Chủ đề: Design, Hash Table, String, Trie, Depth-First Search

Skeleton LeetCode:

```python
class MagicDictionary:

    def __init__(self):
        ...

    def buildDict(self, dictionary: List[str]) -> None:
        ...

    def search(self, searchWord: str) -> bool:
        ...


# Your MagicDictionary object will be instantiated and called as such:
# obj = MagicDictionary()
# obj.buildDict(dictionary)
# param_2 = obj.search(searchWord)
```

Ghi chú: đây là bài **thiết kế lớp** — LeetCode đưa một **chuỗi lệnh** (`["MagicDictionary","buildDict","search",...]`) cùng **chuỗi tham số** và so sánh chuỗi kết quả. Notebook tự dựng hàm `run_commands` mô phỏng cách LeetCode gọi. Ví dụ 2 trong mục 2 là **tự dựng** (đáp án đã kiểm chứng bằng code), nên đối chiếu lại đề gốc nếu cần.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thiết kế lớp `MagicDictionary` nhận một danh sách từ khác nhau (`buildDict`), rồi trả lời `search(word)`: có thể **đổi đúng một ký tự** của `word` để thành **một từ trong từ điển** không? (đổi **đúng một** — không phải 0, không phải 2.)
2. Nắm các công cụ Python: cắt/ghép chuỗi `w[:i] + c + w[i+1:]`, `zip` + `sum` để đếm vị trí khác nhau, `set` / `dict` / `Counter`, `string.ascii_lowercase`, **trie** viết bằng `dict` lồng nhau, DFS với tham số, và khác biệt giữa trạng thái **của đối tượng** (`self.x` trong `__init__`) với trạng thái **của lớp**.
3. Đi từ **so từng từ với `searchWord` (đếm ký tự khác)** → **mẫu thay ký tự bằng `*` + đếm** → **tập mọi "hàng xóm" tạo trước** → **thử thay từng ký tự của `searchWord` rồi tra tập** → **Trie + DFS với biến `diff`**.
4. Hiểu mấu chốt: hai từ **cùng độ dài** khác nhau **đúng một** vị trí ⇔ hai từ **trùng nhau sau khi che đúng vị trí đó bằng `*`**. Và phải xử lý riêng trường hợp "tự khớp với chính nó" (từ có trong từ điển nhưng không có từ nào khác sai một chữ).
5. Tránh các lỗi kinh điển: coi khớp **chính xác** là `True` (đề đòi đổi đúng 1 ký tự), so sánh khi **độ dài khác nhau**, lỗi `zip` cắt cụt, dùng `diff <= 1` thay `== 1`, biến trạng thái ở **mức class**, và quên kiểm tra **kết thúc từ** trong trie.

## 2. Đề bài & ví dụ

Thiết kế một cấu trúc dữ liệu được khởi tạo bằng một danh sách các từ **khác nhau**. Cho một chuỗi, hãy xác định xem có thể **đổi đúng một ký tự** của chuỗi đó để khớp với **bất kỳ** từ nào trong cấu trúc dữ liệu hay không.

Cài đặt lớp `MagicDictionary`:

- `MagicDictionary()` khởi tạo đối tượng.
- `void buildDict(String[] dictionary)` thiết lập cấu trúc dữ liệu với mảng chuỗi `dictionary` (các từ khác nhau).
- `bool search(String searchWord)` trả về `true` nếu có thể đổi **đúng một ký tự** của `searchWord` để thành một chuỗi trong cấu trúc, ngược lại trả `false`.

**Ví dụ 1**

```
Input
["MagicDictionary", "buildDict", "search", "search", "search", "search"]
[[], [["hello", "leetcode"]], ["hello"], ["hhllo"], ["hell"], ["leetcoded"]]
Output
[null, null, false, true, false, false]
```

Giải thích:
- `search("hello")` → `false`: `hello` đã là một từ trong từ điển, nhưng ta phải đổi **đúng một** ký tự (đổi xong phải khác `hello`); không từ nào khác cách `hello` một ký tự.
- `search("hhllo")` → `true`: đổi `h` thứ hai thành `e` được `hello`.
- `search("hell")` → `false`: độ dài `4`, không từ nào dài `4`.
- `search("leetcoded")` → `false`: độ dài `9`, không từ nào dài `9`.

**Ví dụ 2 (tự dựng, đối chiếu lại đề gốc nếu cần)**

```
dictionary = ["abc", "abd", "xyz"]
search("abx") -> true    (abx -> abc hoặc abd)
search("abc") -> true    (abc -> abd: đổi c thành d)
search("xyz") -> false   (không từ nào khác cách xyz đúng một ký tự)
search("abcd") -> false  (độ dài 4)
search("xyy") -> true    (xyy -> xyz)
```

**Ràng buộc:**

- `1 <= dictionary.length <= 100`
- `1 <= dictionary[i].length <= 100`
- `dictionary[i]` chỉ gồm chữ cái thường a–z.
- Các chuỗi trong `dictionary` **khác nhau từng đôi một**.
- `1 <= searchWord.length <= 100`, chỉ gồm chữ thường.
- `buildDict` chỉ được gọi **một lần** trước mọi `search`.
- Tối đa `100` lần gọi `search`.

In [1]:
from typing import List
from collections import Counter
from string import ascii_lowercase
import random
import sys


def run_commands(factory, ops, args):
    """Mô phỏng LeetCode: ops[0] là tên lớp (tạo đối tượng), các lệnh sau gọi method."""
    obj = None
    out = []
    for op, a in zip(ops, args):
        if op == "MagicDictionary":
            obj = factory(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


ops1 = ["MagicDictionary", "buildDict", "search", "search", "search", "search"]
args1 = [[], [["hello", "leetcode"]], ["hello"], ["hhllo"], ["hell"], ["leetcoded"]]
exp1 = [None, None, False, True, False, False]

ops2 = ["MagicDictionary", "buildDict", "search", "search", "search", "search", "search"]
args2 = [[], [["abc", "abd", "xyz"]], ["abx"], ["abc"], ["xyz"], ["abcd"], ["xyy"]]
exp2 = [None, None, True, True, False, False, True]

examples = [(ops1, args1, exp1), (ops2, args2, exp2)]
for ops, args, exp in examples:
    print("lệnh :", ops)
    print("kỳ vọng:", exp)

lệnh : ['MagicDictionary', 'buildDict', 'search', 'search', 'search', 'search']
kỳ vọng: [None, None, False, True, False, False]
lệnh : ['MagicDictionary', 'buildDict', 'search', 'search', 'search', 'search', 'search']
kỳ vọng: [None, None, True, True, False, False, True]


## 3. Kiến thức nền

### 3.1. Chuỗi bất biến: cắt và ghép để "đổi một ký tự"

Chuỗi Python **không sửa được tại chỗ** (`s[1] = 'x'` báo lỗi). Muốn "đổi ký tự thứ `i`" ta **tạo chuỗi mới**: phần trước `s[:i]` + ký tự mới + phần sau `s[i+1:]`. (`s[:i]` lấy `i` ký tự đầu; `s[i+1:]` lấy từ sau vị trí `i` đến hết.)

In [2]:
s = "hhllo"
try:
    s[1] = "e"
except TypeError as e:
    print("không sửa tại chỗ được:", e)

i = 1
print("trước:", s[:i], "| sau:", s[i + 1:])
print("đổi ký tự 1 thành 'e':", s[:i] + "e" + s[i + 1:])
print("che ký tự 1 bằng '*'  :", s[:i] + "*" + s[i + 1:])

không sửa tại chỗ được: 'str' object does not support item assignment
trước: h | sau: llo
đổi ký tự 1 thành 'e': hello
che ký tự 1 bằng '*'  : h*llo


### 3.2. Đếm số vị trí khác nhau: `zip` và `sum`

`zip(a, b)` ghép ký tự cùng vị trí. `a_k != b_k` cho `True`/`False`; trong Python `True` bằng `1`, `False` bằng `0`, nên `sum(x != y for x, y in zip(a, b))` đếm số vị trí khác nhau — gọi là **khoảng cách Hamming** (chỉ đúng khi hai chuỗi **cùng độ dài**; `zip` cắt theo chuỗi **ngắn hơn**).

In [3]:
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

print(hamming("hello", "hhllo"))    # 1
print(hamming("hello", "hello"))    # 0
print(hamming("hello", "world"))    # 4
print(hamming("hxl", "hello"))      # zip cắt cụt: chỉ so 3 cặp đầu -> kết quả gây hiểu lầm!

1
0
4
1


Điều cuối cùng là **bẫy**: `hamming("hxl", "hello")` bằng `1` dù hai chuỗi **không cùng độ dài**. Luôn kiểm tra `len(a) == len(b)` trước.

### 3.3. `set`, `dict`, `Counter`

- `set`: tập hợp, kiểm tra `x in s` trung bình O(1) (băm chuỗi tốn O(độ dài chuỗi)).
- `dict.get(k, 0)`: đọc giá trị, mặc định `0` nếu thiếu khóa — **không** tạo khóa.
- `Counter`: dict đếm; `cnt[x] += 1`. Đọc khóa thiếu bằng `cnt[x]` trả `0` (Counter **không** tạo khóa khi đọc kiểu này).

In [4]:
words = {"hello", "leetcode"}
print("hello" in words, "hell" in words)

cnt = Counter()
for p in ["h*llo", "h*llo", "he*lo"]:
    cnt[p] += 1
print(dict(cnt))
print("khóa thiếu:", cnt["zzz"], "| có bị tạo không?", "zzz" in cnt)

d = {}
print(d.get("a", 0), "a" in d)

True False
{'h*llo': 2, 'he*lo': 1}
khóa thiếu: 0 | có bị tạo không? False
0 False


### 3.4. `string.ascii_lowercase`

Chuỗi `'abcdefghijklmnopqrstuvwxyz'` (26 chữ thường) — tiện để thử thay một ký tự bằng mọi chữ cái.

In [5]:
print(ascii_lowercase, len(ascii_lowercase))
w = "cat"
i = 1
print([w[:i] + c + w[i + 1:] for c in ascii_lowercase if c != w[i]][:5], "...")

abcdefghijklmnopqrstuvwxyz 26
['cbt', 'cct', 'cdt', 'cet', 'cft'] ...


### 3.5. Trie bằng `dict` lồng nhau

**Trie** (cây tiền tố) lưu các từ sao cho các từ **chung tiền tố** dùng chung đường đi. Mỗi nút là một `dict`: khóa là ký tự, giá trị là nút con. Ta đánh dấu "một từ kết thúc ở đây" bằng khóa đặc biệt `'#'` (không phải chữ thường nên không đụng chạm). `node.setdefault(ch, {})` lấy nút con của `ch`, nếu chưa có thì tạo `{}`.

In [6]:
root = {}
for w in ["hello", "help", "leet"]:
    node = root
    for ch in w:
        node = node.setdefault(ch, {})
    node["#"] = True

import json
print(json.dumps(root))
print("h->e->l có các nhánh tiếp:", list(root["h"]["e"]["l"].keys()))

{"h": {"e": {"l": {"l": {"o": {"#": true}}, "p": {"#": true}}}}, "l": {"e": {"e": {"t": {"#": true}}}}}
h->e->l có các nhánh tiếp: ['l', 'p']


### 3.6. DFS có tham số

DFS (duyệt theo chiều sâu) viết bằng đệ quy: hàm `dfs(node, i, diff)` nhận nút trie hiện tại, vị trí `i` đang xét trong `searchWord`, và số ký tự đã đổi `diff`. Mỗi lời gọi thử các nhánh con; trả `True` nếu **một** nhánh dẫn tới thành công (dùng `if dfs(...): return True`).

Mẫu quen thuộc: "thử đi theo ký tự khớp; hoặc dùng quyền đổi (nếu còn)".

### 3.7. Trạng thái của đối tượng và của lớp

Biến gán trong `__init__` bằng `self.x = ...` là của **từng đối tượng**. Biến khai báo ngay trong thân `class` (ngoài mọi method) là của **cả lớp**, **dùng chung** cho mọi đối tượng — với kiểu **thay đổi được** (list, set, dict) sẽ gây lỗi khó thấy. Ta sẽ gặp lỗi này ở mục 13.

### 3.8. Mô phỏng cách LeetCode chạy bài thiết kế lớp

LeetCode nhận hai danh sách song song: danh sách tên lệnh (`ops`) và danh sách tham số (`args`). Lệnh đầu là tên lớp (tạo đối tượng), các lệnh sau gọi method trên đối tượng đó; kết quả của hàm không trả giá trị là `null` (`None` trong Python). Hàm `run_commands` ở trên làm đúng việc ấy: ta truyền vào một "nhà máy" `factory` (chính là lớp) để dùng được cho nhiều cài đặt khác nhau.

## 4. Ý tưởng nền tảng / trực giác

Với từ điển `["hello", "leetcode"]` và `searchWord = "hhllo"`, so từng cặp ký tự với `"hello"`:

| vị trí | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| `hhllo` | h | h | l | l | o |
| `hello` | h | e | l | l | o |
| khác? | | **khác** | | | |

Đúng **một** vị trí khác → trả `True`. Với `searchWord = "hello"`: **không có** vị trí khác (0 vị trí) → không đủ "đúng một" → `False` (trừ khi có từ khác cách `hello` đúng 1 ký tự).

Vậy: `search(w)` đúng ⇔ **tồn tại** từ `x` trong từ điển sao cho `len(x) == len(w)` và `hamming(x, w) == 1`.

Ta viết ngay "trọng tài" `oracle_search` theo đúng định nghĩa này — ngắn, không tối ưu, để đối chiếu mọi cách giải.

In [7]:
def oracle_search(dictionary, w):
    return any(len(x) == len(w) and sum(a != b for a, b in zip(x, w)) == 1 for x in dictionary)

d = ["hello", "leetcode"]
for w in ["hello", "hhllo", "hell", "leetcoded"]:
    print(f"{w!r:12} ->", oracle_search(d, w))

assert [oracle_search(d, w) for w in ["hello", "hhllo", "hell", "leetcoded"]] == [False, True, False, False]
d2 = ["abc", "abd", "xyz"]
assert [oracle_search(d2, w) for w in ["abx", "abc", "xyz", "abcd", "xyy"]] == [True, True, False, False, True]
print("oracle khớp hai ví dụ")

'hello'      -> False
'hhllo'      -> True
'hell'       -> False
'leetcoded'  -> False
oracle khớp hai ví dụ


## 5. Cách 1 — Brute Force: so `searchWord` với từng từ trong từ điển

**Ý tưởng.** `buildDict` chỉ lưu lại danh sách từ. `search(w)` duyệt từng từ `x`:

1. Bỏ qua nếu `len(x) != len(w)`.
2. Đếm số vị trí khác nhau `diff`; nếu `diff` vượt `1` thì dừng sớm.
3. Nếu `diff == 1` → trả `True`. Hết danh sách → `False`.

Đây là cài đặt "đọc đề viết code". Cái ta cần làm xong là một lớp có cùng giao diện để `run_commands` gọi được.

In [8]:
class MagicDictionaryBrute:
    def __init__(self):
        self.words = []

    def buildDict(self, dictionary):
        self.words = list(dictionary)

    def search(self, searchWord):
        for x in self.words:
            if len(x) != len(searchWord):
                continue
            diff = 0
            for a, b in zip(x, searchWord):
                if a != b:
                    diff += 1
                    if diff > 1:
                        break
            if diff == 1:
                return True
        return False

for ops, args, exp in examples:
    got = run_commands(MagicDictionaryBrute, ops, args)
    assert got == exp, (got, exp)
    print(got)

[None, None, False, True, False, False]
[None, None, True, True, False, False, True]


**Độ phức tạp.** Gọi `N` là số từ, `L` là độ dài tối đa. `buildDict`: O(N). Mỗi `search`: duyệt `N` từ, mỗi từ so tối đa `L` ký tự → **O(N·L)**. Với `N, L ≤ 100` và ≤ 100 lần gọi: khoảng `10^6` thao tác — đủ nhanh ở đây. **Space O(N·L)** để lưu từ điển.

**Nút thắt.** Mỗi truy vấn đều quét **toàn bộ** từ điển, kể cả những từ chắc chắn không khớp, và so từng ký tự. Nếu có `10^5` từ thì mỗi `search` rất chậm. Ta muốn **tra thẳng** thay vì quét: đó là bước đột phá ở mục tiếp theo.

## 6. Ý tưởng thuật toán / bước đột phá

Hai chuỗi cùng độ dài khác nhau **đúng một vị trí `i`** ⇔ sau khi **che vị trí `i` bằng `*`**, hai chuỗi **giống hệt nhau**:

```
hhllo  ->  h*llo
hello  ->  h*llo      (che vị trí 1)   => giống nhau => hai từ chỉ khác nhau ở vị trí 1
```

Vậy: với mỗi từ trong từ điển, tạo `L` **mẫu** (pattern) — mỗi mẫu che một vị trí — và đưa vào bảng băm. Với `searchWord`, cũng tạo `L` mẫu và **tra bảng**: nếu mẫu nào xuất hiện thì có từ trong từ điển trùng mẫu ở vị trí đó.

Nhưng có **một bẫy**: mẫu của `searchWord` cũng khớp mẫu của **chính nó** nếu `searchWord` có trong từ điển (hello → `h*llo` khớp `h*llo` của hello — dù hai từ giống hệt nhau, khác 0 vị trí). Cách xử lý: **đếm** số từ cho mỗi mẫu bằng `Counter`. Với mẫu `p` của `w`:

- Số từ khớp `p` là `c`. Nếu `w` **có** trong từ điển thì **một** trong `c` từ chính là `w` — còn `c - 1` từ khác.
- Nếu `w` **không** có trong từ điển thì cả `c` từ đều khác `w`.

→ tồn tại từ **khác `w`** khớp mẫu ⇔ `c > 1`, hoặc `c == 1` và `w` **không** nằm trong từ điển.

Hãy xem các mẫu và số đếm trên ví dụ nhỏ.

In [9]:
def patterns(w):
    return [w[:i] + "*" + w[i + 1:] for i in range(len(w))]

dictionary = ["hello", "hallo", "leetcode"]
cnt = Counter(p for w in dictionary for p in patterns(w))
print("patterns('hello') =", patterns("hello"))
print("cnt['h*llo'] =", cnt["h*llo"], "(hello và hallo)")
print("cnt['he*lo'] =", cnt["he*lo"], "(chỉ hello)")

words = set(dictionary)
for w in ["hello", "hxllo", "hella", "leetcode"]:
    hit = False
    for p in patterns(w):
        c = cnt.get(p, 0)
        if c > 1 or (c == 1 and w not in words):
            hit = True
            break
    print(f"{w!r:12} ->", hit, "| oracle:", oracle_search(dictionary, w))
    assert hit == oracle_search(dictionary, w)

patterns('hello') = ['*ello', 'h*llo', 'he*lo', 'hel*o', 'hell*']
cnt['h*llo'] = 2 (hello và hallo)
cnt['he*lo'] = 1 (chỉ hello)
'hello'      -> True | oracle: True
'hxllo'      -> True | oracle: True
'hella'      -> True | oracle: True
'leetcode'   -> False | oracle: False


Chú ý `hello`: mẫu `h*llo` có `c = 2` (vì có cả `hallo`) → `True`; còn `leetcode` mọi mẫu chỉ có `c = 1` mà `leetcode` lại có trong từ điển → `False`.

## 7. Các cách tiếp theo

5 cách tổng cộng (cách 1 đã ở mục 5). Gọi `N` số từ, `L` độ dài tối đa:

| # | Cách | `buildDict` | `search` | Space |
|---|---|---|---|---|
| 1 | So từng từ (brute force) | O(N) | O(N·L) | O(N·L) |
| 2 | Mẫu `*` + `Counter` | O(N·L²) | O(L²) | O(N·L²) |
| 3 | Tập mọi "hàng xóm" tạo trước | O(25·N·L²) | O(L) | O(25·N·L²) |
| 4 | Thử thay từng ký tự của `searchWord` | O(N·L) | O(26·L²) | O(N·L) |
| 5 | Trie + DFS | O(N·L) | O(26·L²) tệ nhất | O(N·L) |

(Các hệ số `L²` do mỗi lần tạo chuỗi mới tốn O(L).)

### 7.1. Cách 2 — Mẫu `*` và `Counter`

**Giải thích từng đoạn.**

1. `buildDict`: lưu `self.words = set(dictionary)` (để biết `w` có trong từ điển không) và `self.cnt` = số từ theo từng mẫu.
2. `search(w)`: với mỗi vị trí `i`, mẫu `p = w[:i] + '*' + w[i+1:]`; lấy `c = self.cnt.get(p, 0)`; nếu `c > 1` hoặc (`c == 1` và `w` không có trong từ điển) → `True`.
3. Không tìm thấy → `False`.

Mẫu của từ khác độ dài không bao giờ trùng (độ dài chuỗi khác nhau), nên không cần kiểm tra độ dài riêng.

In [10]:
class MagicDictionaryPattern:
    def __init__(self):
        self.words = set()
        self.cnt = {}

    def buildDict(self, dictionary):
        self.words = set(dictionary)
        self.cnt = {}
        for w in dictionary:
            for i in range(len(w)):
                p = w[:i] + "*" + w[i + 1:]
                self.cnt[p] = self.cnt.get(p, 0) + 1

    def search(self, searchWord):
        w = searchWord
        for i in range(len(w)):
            p = w[:i] + "*" + w[i + 1:]
            c = self.cnt.get(p, 0)
            if c > 1 or (c == 1 and w not in self.words):
                return True
        return False

for ops, args, exp in examples:
    got = run_commands(MagicDictionaryPattern, ops, args)
    assert got == exp, (got, exp)
    print(got)

[None, None, False, True, False, False]
[None, None, True, True, False, False, True]


In [11]:
# Code nộp LeetCode — Cách 2: mẫu '*' + đếm
class MagicDictionary:

    def __init__(self):
        self.words = set()
        self.cnt = {}

    def buildDict(self, dictionary: List[str]) -> None:
        self.words = set(dictionary)
        self.cnt = {}
        for w in dictionary:
            for i in range(len(w)):
                p = w[:i] + "*" + w[i + 1:]
                self.cnt[p] = self.cnt.get(p, 0) + 1

    def search(self, searchWord: str) -> bool:
        w = searchWord
        for i in range(len(w)):
            p = w[:i] + "*" + w[i + 1:]
            c = self.cnt.get(p, 0)
            if c > 1 or (c == 1 and w not in self.words):
                return True
        return False

for ops, args, exp in examples:
    assert run_commands(MagicDictionary, ops, args) == exp
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp.** `buildDict`: mỗi từ tạo `L` mẫu, mỗi mẫu tốn O(L) (tạo chuỗi + băm) → **O(N·L²)**. `search`: `L` mẫu × O(L) → **O(L²)** — **không phụ thuộc `N`**. **Space O(N·L²)** cho các mẫu (tối đa `10^6` ký tự với `N = L = 100`).

### 7.2. Cách 3 — Tạo trước tập mọi "hàng xóm" (build-side)

**Ý tưởng khác.** Thay vì đếm mẫu, ta **trực tiếp** liệt kê, với mỗi từ `x`, **mọi chuỗi cách `x` đúng một ký tự**: với mỗi vị trí `i` và mỗi chữ cái `c ≠ x[i]` thì thêm `x[:i] + c + x[i+1:]` vào tập `near`. Khi đó `search(w)` chỉ là `w in near` — `True` ⇔ `w` cách **ít nhất một** từ điển đúng một ký tự.

Vì ta **loại** `c == x[i]` nên `x` không tự thêm chính nó; và nếu `w` trùng một từ khác `y` trong từ điển mà cách `x` đúng một ký tự thì `w ∈ near(x)` — đúng (`w = y` nhưng đổi `y` một ký tự được `x`).

Giá phải trả: mỗi từ sinh `25 · L` chuỗi dài `L` → tốn bộ nhớ và thời gian dựng, nhưng `search` cực nhanh.

In [12]:
class MagicDictionaryNear:
    def __init__(self):
        self.near = set()

    def buildDict(self, dictionary):
        self.near = set()
        for x in dictionary:
            for i, ch in enumerate(x):
                for c in ascii_lowercase:
                    if c != ch:
                        self.near.add(x[:i] + c + x[i + 1:])

    def search(self, searchWord):
        return searchWord in self.near

for ops, args, exp in examples:
    got = run_commands(MagicDictionaryNear, ops, args)
    assert got == exp, (got, exp)
    print(got)

m = MagicDictionaryNear()
m.buildDict(["hello", "leetcode"])
print("số hàng xóm đã tạo:", len(m.near), "= 25 * (5 + 8) =", 25 * 13)

[None, None, False, True, False, False]
[None, None, True, True, False, False, True]
số hàng xóm đã tạo: 325 = 25 * (5 + 8) = 325


In [13]:
# Code nộp LeetCode — Cách 3: tập mọi hàng xóm
class MagicDictionary:

    def __init__(self):
        self.near = set()

    def buildDict(self, dictionary: List[str]) -> None:
        self.near = set()
        for x in dictionary:
            for i, ch in enumerate(x):
                for c in "abcdefghijklmnopqrstuvwxyz":
                    if c != ch:
                        self.near.add(x[:i] + c + x[i + 1:])

    def search(self, searchWord: str) -> bool:
        return searchWord in self.near

for ops, args, exp in examples:
    assert run_commands(MagicDictionary, ops, args) == exp
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp.** `buildDict`: `N·L·25` chuỗi, mỗi chuỗi O(L) → **O(25·N·L²)** (với `N = L = 100`: 250 nghìn chuỗi dài 100 ≈ vài chục MB, vẫn chấp nhận được ở ràng buộc này). `search`: một lần tra tập → **O(L)** (băm chuỗi). **Space O(25·N·L²)**. Cách này đổi **bộ nhớ lấy tốc độ truy vấn**.

### 7.3. Cách 4 — Thử thay từng ký tự của `searchWord` rồi tra tập (query-side)

**Ý tưởng đối xứng** với cách 3: giữ từ điển là một `set` thường (nhỏ gọn), còn khi truy vấn thì **tự sinh** hàng xóm của `searchWord`: với mỗi vị trí `i` và mỗi chữ cái `c ≠ w[i]`, nếu `w[:i] + c + w[i+1:]` nằm trong `set` thì có từ trong từ điển cách `w` đúng một ký tự → `True`.

Vì `c ≠ w[i]` nên chuỗi sinh ra **luôn khác `w`**; không bao giờ coi `w` là hàng xóm của chính nó → không cần xử lý "tự khớp". Mỗi truy vấn thử `25·L` chuỗi, mỗi chuỗi O(L).

In [14]:
class MagicDictionaryQuery:
    def __init__(self):
        self.words = set()

    def buildDict(self, dictionary):
        self.words = set(dictionary)

    def search(self, searchWord):
        w = searchWord
        for i, ch in enumerate(w):
            for c in ascii_lowercase:
                if c != ch and w[:i] + c + w[i + 1:] in self.words:
                    return True
        return False

for ops, args, exp in examples:
    got = run_commands(MagicDictionaryQuery, ops, args)
    assert got == exp, (got, exp)
    print(got)

[None, None, False, True, False, False]
[None, None, True, True, False, False, True]


In [15]:
# Code nộp LeetCode — Cách 4: thử thay từng ký tự của searchWord
class MagicDictionary:

    def __init__(self):
        self.words = set()

    def buildDict(self, dictionary: List[str]) -> None:
        self.words = set(dictionary)

    def search(self, searchWord: str) -> bool:
        w = searchWord
        for i, ch in enumerate(w):
            for c in "abcdefghijklmnopqrstuvwxyz":
                if c != ch and w[:i] + c + w[i + 1:] in self.words:
                    return True
        return False

for ops, args, exp in examples:
    assert run_commands(MagicDictionary, ops, args) == exp
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp.** `buildDict`: **O(N·L)** (băm từng từ). `search`: `L` vị trí × 25 chữ cái × O(L) tạo/băm chuỗi → **O(25·L²)** — độc lập với `N`. **Space O(N·L)** — nhỏ nhất trong các cách tra bảng băm.

### 7.4. Cách 5 — Trie + DFS với biến `diff`

**Ý tưởng.** Lưu từ điển trong **trie**. `search(w)` duyệt trie song song với `w` bằng DFS `dfs(node, i, diff)`:

- `i` là vị trí đang xét trong `w`, `diff` là số ký tự đã đổi (0 hoặc 1).
- Nếu `i == len(w)`: thành công **khi và chỉ khi** `diff == 1` **và** nút hiện tại là **kết thúc của một từ** (`'#' in node`).
- Ngược lại duyệt từng con `(ch, child)` của `node` (bỏ khóa `'#'`): `d = diff + (ch != w[i])`; chỉ đi tiếp nếu `d <= 1`.

Khi `diff == 1` (đã dùng quyền đổi) ta chỉ cần đi theo **đúng** ký tự `w[i]` (các ký tự khác sẽ làm `d = 2`) nên rẽ nhánh thừa bị cắt ngay.

Ưu điểm: các từ chung tiền tố **dùng chung** nút → ít bộ nhớ, và rẽ nhánh trie thường thưa nên nhanh trong thực tế.

In [16]:
class MagicDictionaryTrie:
    def __init__(self):
        self.root = {}

    def buildDict(self, dictionary):
        self.root = {}
        for w in dictionary:
            node = self.root
            for ch in w:
                node = node.setdefault(ch, {})
            node["#"] = True

    def search(self, searchWord):
        w = searchWord
        n = len(w)

        def dfs(node, i, diff):
            if i == n:
                return diff == 1 and "#" in node
            if diff == 1:                       # hết quyền đổi: chỉ đi theo đúng ký tự
                child = node.get(w[i])
                return child is not None and dfs(child, i + 1, 1)
            for ch, child in node.items():
                if ch == "#":
                    continue
                if dfs(child, i + 1, 0 if ch == w[i] else 1):
                    return True
            return False

        return dfs(self.root, 0, 0)

for ops, args, exp in examples:
    got = run_commands(MagicDictionaryTrie, ops, args)
    assert got == exp, (got, exp)
    print(got)

[None, None, False, True, False, False]
[None, None, True, True, False, False, True]


In [17]:
# Code nộp LeetCode — Cách 5: Trie + DFS
class MagicDictionary:

    def __init__(self):
        self.root = {}

    def buildDict(self, dictionary: List[str]) -> None:
        self.root = {}
        for w in dictionary:
            node = self.root
            for ch in w:
                node = node.setdefault(ch, {})
            node["#"] = True

    def search(self, searchWord: str) -> bool:
        w = searchWord
        n = len(w)

        def dfs(node, i, diff):
            if i == n:
                return diff == 1 and "#" in node
            if diff == 1:
                child = node.get(w[i])
                return child is not None and dfs(child, i + 1, 1)
            for ch, child in node.items():
                if ch == "#":
                    continue
                if dfs(child, i + 1, 0 if ch == w[i] else 1):
                    return True
            return False

        return dfs(self.root, 0, 0)

for ops, args, exp in examples:
    assert run_commands(MagicDictionary, ops, args) == exp
print("Cách 5 qua ví dụ")

Cách 5 qua ví dụ


**Độ phức tạp.** `buildDict`: **O(N·L)** (mỗi ký tự một bước `setdefault`). `search`: tại mỗi vị trí `i` (khi `diff = 0`) ta thử tối đa 26 nhánh; mỗi nhánh đổi (`diff` thành 1) đi tiếp theo đường duy nhất tối đa `L - i` bước → tệ nhất **O(26·L²)**, nhưng thực tế nhỏ hơn rất nhiều vì trie hiếm khi đầy đủ nhánh. Độ sâu đệ quy `≤ L = 100`, an toàn. **Space O(N·L)** nút trie tối đa.

## 8. TRACE từng bước

**Trace cách 2 (mẫu `*`)** với từ điển `["hello", "hallo"]`. Bảng `cnt` sau `buildDict`:

| mẫu | `cnt` |
|---|---|
| `*ello` | 1 (hello) |
| `h*llo` | **2** (hello, hallo) |
| `he*lo` | 1 |
| `hel*o` | 1 |
| `hell*` | 1 |
| `*allo` | 1 (hallo) |
| `ha*lo`, `hal*o`, `hall*` | 1 mỗi mẫu |

`search("hello")` (có trong từ điển): `i = 0`: `*ello` có `c = 1`, `hello` ∈ từ điển → bỏ; `i = 1`: `h*llo` có `c = 2 > 1` → **True** (vì còn `hallo`).

Với từ điển chỉ `["hello"]` thì `h*llo` có `c = 1` và `hello` ∈ từ điển → không có từ **khác**; cả 5 vị trí đều bỏ → **False**.

In [18]:
def trace_pattern(dictionary, w):
    words = set(dictionary)
    cnt = {}
    for x in dictionary:
        for i in range(len(x)):
            p = x[:i] + "*" + x[i + 1:]
            cnt[p] = cnt.get(p, 0) + 1
    print("từ điển:", dictionary, "| search:", repr(w))
    print("cnt =", cnt)
    for i in range(len(w)):
        p = w[:i] + "*" + w[i + 1:]
        c = cnt.get(p, 0)
        in_dict = w in words
        ok = c > 1 or (c == 1 and not in_dict)
        print(f"i={i}: mẫu={p!r:9} c={c} w có trong từ điển? {in_dict} -> {'KHỚP' if ok else 'không'}")
        if ok:
            print("=> True")
            return True
    print("=> False")
    return False

assert trace_pattern(["hello", "hallo"], "hello") is True
print()
assert trace_pattern(["hello"], "hello") is False
print()
assert trace_pattern(["hello", "leetcode"], "hhllo") is True

từ điển: ['hello', 'hallo'] | search: 'hello'
cnt = {'*ello': 1, 'h*llo': 2, 'he*lo': 1, 'hel*o': 1, 'hell*': 1, '*allo': 1, 'ha*lo': 1, 'hal*o': 1, 'hall*': 1}
i=0: mẫu='*ello'   c=1 w có trong từ điển? True -> không
i=1: mẫu='h*llo'   c=2 w có trong từ điển? True -> KHỚP
=> True

từ điển: ['hello'] | search: 'hello'
cnt = {'*ello': 1, 'h*llo': 1, 'he*lo': 1, 'hel*o': 1, 'hell*': 1}
i=0: mẫu='*ello'   c=1 w có trong từ điển? True -> không
i=1: mẫu='h*llo'   c=1 w có trong từ điển? True -> không
i=2: mẫu='he*lo'   c=1 w có trong từ điển? True -> không
i=3: mẫu='hel*o'   c=1 w có trong từ điển? True -> không
i=4: mẫu='hell*'   c=1 w có trong từ điển? True -> không
=> False

từ điển: ['hello', 'leetcode'] | search: 'hhllo'
cnt = {'*ello': 1, 'h*llo': 1, 'he*lo': 1, 'hel*o': 1, 'hell*': 1, '*eetcode': 1, 'l*etcode': 1, 'le*tcode': 1, 'lee*code': 1, 'leet*ode': 1, 'leetc*de': 1, 'leetco*e': 1, 'leetcod*': 1}
i=0: mẫu='*hllo'   c=0 w có trong từ điển? False -> không
i=1: mẫu='h*llo'   c=1 w

**Trace cách 5 (Trie + DFS)** cho `searchWord = "hhllo"` với từ điển `["hello", "leetcode"]`. Trie có hai nhánh gốc: `h` (→ e → l → l → o → `#`) và `l` (→ e → e → t → ...).

| bước | nút (đường đi) | i | ký tự `w[i]` | thử nhánh `ch` | `diff` mới | kết quả |
|---|---|---|---|---|---|---|
| 1 | gốc | 0 | h | `h` | 0 | đi tiếp |
| 2 | `h` | 1 | h | `e` | 1 (đổi) | đi tiếp (đã dùng quyền đổi) |
| 3 | `he` | 2 | l | `l` (chỉ đi theo `w[i]`) | 1 | đi tiếp |
| 4 | `hel` | 3 | l | `l` | 1 | đi tiếp |
| 5 | `hell` | 4 | o | `o` | 1 | đi tiếp |
| 6 | `hello` | 5 | — | `i == n` | 1 | `diff == 1` và `'#' in node` → **True** |

Nhánh `l` (của `leetcode`) ở gốc không bao giờ được thử vì DFS đã trả `True` trước.

In [19]:
def trace_trie(dictionary, w):
    root = {}
    for x in dictionary:
        node = root
        for ch in x:
            node = node.setdefault(ch, {})
        node["#"] = True
    n = len(w)
    print("từ điển:", dictionary, "| search:", repr(w))

    def dfs(node, i, diff, path):
        pad = "  " * i
        if i == n:
            ok = diff == 1 and "#" in node
            print(f"{pad}i={i} hết chuỗi tại '{path}': diff={diff}, kết thúc từ? {'#' in node} -> {ok}")
            return ok
        if diff == 1:
            child = node.get(w[i])
            print(f"{pad}i={i} đường '{path}' diff=1: chỉ đi theo {w[i]!r} -> {'có' if child is not None else 'không có nhánh'}")
            return child is not None and dfs(child, i + 1, 1, path + w[i])
        for ch, child in node.items():
            if ch == "#":
                continue
            nd = 0 if ch == w[i] else 1
            print(f"{pad}i={i} đường '{path}' thử nhánh {ch!r} (w[i]={w[i]!r}) -> diff={nd}")
            if dfs(child, i + 1, nd, path + ch):
                return True
        print(f"{pad}i={i} đường '{path}': hết nhánh -> False")
        return False

    res = dfs(root, 0, 0, "")
    print("kết quả:", res)
    return res

assert trace_trie(["hello", "leetcode"], "hhllo") is True
print()
assert trace_trie(["hello", "leetcode"], "hello") is False

từ điển: ['hello', 'leetcode'] | search: 'hhllo'
i=0 đường '' thử nhánh 'h' (w[i]='h') -> diff=0
  i=1 đường 'h' thử nhánh 'e' (w[i]='h') -> diff=1
    i=2 đường 'he' diff=1: chỉ đi theo 'l' -> có
      i=3 đường 'hel' diff=1: chỉ đi theo 'l' -> có
        i=4 đường 'hell' diff=1: chỉ đi theo 'o' -> có
          i=5 hết chuỗi tại 'hello': diff=1, kết thúc từ? True -> True
kết quả: True

từ điển: ['hello', 'leetcode'] | search: 'hello'
i=0 đường '' thử nhánh 'h' (w[i]='h') -> diff=0
  i=1 đường 'h' thử nhánh 'e' (w[i]='e') -> diff=0
    i=2 đường 'he' thử nhánh 'l' (w[i]='l') -> diff=0
      i=3 đường 'hel' thử nhánh 'l' (w[i]='l') -> diff=0
        i=4 đường 'hell' thử nhánh 'o' (w[i]='o') -> diff=0
          i=5 hết chuỗi tại 'hello': diff=0, kết thúc từ? True -> False
        i=4 đường 'hell': hết nhánh -> False
      i=3 đường 'hel': hết nhánh -> False
    i=2 đường 'he': hết nhánh -> False
  i=1 đường 'h': hết nhánh -> False
i=0 đường '' thử nhánh 'l' (w[i]='h') -> diff=1
  i=1 đườ

Trace thứ hai (`search("hello")`): DFS đi theo `h-e-l-l-o` với `diff = 0` tới cuối: `i == n` nhưng `diff = 0 != 1` → thất bại; quay lui, ở mỗi vị trí thử đổi một ký tự rồi đi theo đúng `w[i]` nhưng đường trie cụt (không có từ nào khác cách `hello` một ký tự). Ví dụ ở gốc thử nhánh `l` (đổi `h` thành `l`, `diff = 1`), đi theo `e` thì được (`le`), nhưng ký tự kế là `l` trong khi trie ở đó chỉ có `e` → không có nhánh → thất bại. Cuối cùng `False`.

## 9. Code nộp LeetCode

Ta nộp **cách 2 (mẫu `*` + đếm)**: truy vấn không phụ thuộc số từ, bộ nhớ vừa phải, logic ngắn gọn dễ giải thích. (Nếu đề đòi bộ nhớ thấp nhất thì cách 4 hoặc trie là lựa chọn.)

In [20]:
class MagicDictionary:

    def __init__(self):
        self.words = set()
        self.cnt = {}

    def buildDict(self, dictionary: List[str]) -> None:
        self.words = set(dictionary)
        self.cnt = {}
        for w in dictionary:
            for i in range(len(w)):
                p = w[:i] + "*" + w[i + 1:]
                self.cnt[p] = self.cnt.get(p, 0) + 1

    def search(self, searchWord: str) -> bool:
        w = searchWord
        for i in range(len(w)):
            p = w[:i] + "*" + w[i + 1:]
            c = self.cnt.get(p, 0)
            if c > 1 or (c == 1 and w not in self.words):
                return True
        return False


for ops, args, exp in examples:
    got = run_commands(MagicDictionary, ops, args)
    print(got, "(kỳ vọng", exp, ")")
    assert got == exp

[None, None, False, True, False, False] (kỳ vọng [None, None, False, True, False, False] )
[None, None, True, True, False, False, True] (kỳ vọng [None, None, True, True, False, False, True] )


## 10. Vì sao đúng

**Tương đương then chốt (cách 2).** Với hai chuỗi `a`, `b` **cùng độ dài**: `a` và `b` khác nhau **đúng một** vị trí `i` ⇔ có `i` sao cho `mask(a, i) == mask(b, i)` **và** `a != b` (với `mask(s, i)` là `s` có vị trí `i` thay bằng `*`).

- (⇒) Nếu chỉ khác tại `i` thì sau khi che `i` hai chuỗi bằng nhau, và `a != b`.
- (⇐) Nếu `mask(a, i) == mask(b, i)` thì `a` và `b` giống nhau ở mọi vị trí `≠ i`; nếu thêm `a != b` thì chỉ có thể khác ở `i` → khác **đúng một** vị trí.

Chuỗi khác độ dài không bao giờ có cùng mẫu (mẫu giữ nguyên độ dài), nên vắng mặt kiểm tra độ dài là an toàn.

**Điều kiện `c > 1 or (c == 1 and w not in words)`.** Gọi `S(p)` là tập các từ trong từ điển có mẫu `p` ở một vị trí. `c = |S(p)|` (mỗi từ đóng góp tối đa một lần cho mỗi mẫu **ở mỗi vị trí**; hai vị trí khác nhau của cùng một từ có thể cho **hai mẫu khác nhau** vì `*` nằm khác chỗ, nên không đếm trùng một từ hai lần trong cùng mẫu). Ta cần `S(p)` chứa một từ **khác `w`**. Nếu `w ∈ từ điển` thì `w ∈ S(p)` (vì `w` có mẫu `p` ở chính vị trí `i`), nên `S(p)` có từ khác `w` ⇔ `c ≥ 2`. Nếu `w ∉ từ điển` thì mọi từ trong `S(p)` đều khác `w` ⇔ `c ≥ 1`. Hợp lại cho điều kiện trong code.

**Cách 3 và 4** đúng do chúng liệt kê **mọi** chuỗi khác một ký tự (qua `c ≠ ch`): `w ∈ near` hoặc một hàng xóm của `w` thuộc `words` ⇔ tồn tại từ `x` cách `w` đúng một ký tự. (Quan hệ "cách đúng một ký tự" đối xứng.)

**Cách 5 (trie + DFS)** — quy nạp theo `i`: `dfs(node, i, diff)` trả `True` ⇔ tồn tại từ `x` đi qua `node` (tiền tố độ dài `i`), `x[i:]` cách `w[i:]` đúng `1 - diff` ký tự và `x` kết thúc ở đúng độ dài `n`. Trường hợp cơ sở `i == n`: cần `diff == 1` và nút là kết thúc từ (để độ dài `x` bằng `n`, nếu không `x` dài hơn). Bước quy nạp: ký tự `i` của `x` hoặc bằng `w[i]` (giữ `diff`) hoặc khác (tăng `diff`, chỉ cho phép khi `diff = 0`); mọi `x` đều được xét vì ta duyệt **mọi** nhánh con cho phép.

**Cách 1** đúng trực tiếp theo định nghĩa (so từng từ cùng độ dài và đếm).

## 11. Time / Space Complexity

`N` số từ, `L` độ dài từ lớn nhất, `Q` số lần `search`.

| Cách | `buildDict` | `search` | Space | Ghi chú |
|---|---|---|---|---|
| 1. So từng từ | O(N) | O(N·L) | O(N·L) | đơn giản; chậm nếu N lớn |
| 2. Mẫu `*` + Counter | O(N·L²) | O(L²) | O(N·L²) | **cách nộp**; truy vấn không phụ thuộc N |
| 3. Tập hàng xóm | O(25·N·L²) | O(L) | O(25·N·L²) | truy vấn nhanh nhất, tốn RAM nhất |
| 4. Thử thay ký tự | O(N·L) | O(25·L²) | O(N·L) | nhỏ gọn, `buildDict` rất nhanh |
| 5. Trie + DFS | O(N·L) | O(26·L²) tệ nhất | O(N·L) | thường nhanh trong thực tế; chung tiền tố |

Với `N, L, Q ≤ 100` mọi cách đều rất nhanh (dưới 1 giây).

## 12. Bộ test

Với bài thiết kế lớp ta kiểm bằng cách **chạy chuỗi lệnh** qua `run_commands` rồi so với kết quả mong đợi (và với `oracle_search`). Gom cả 5 cài đặt vào một dict để đối chiếu.

In [21]:
all_classes = {
    "brute": MagicDictionaryBrute,
    "pattern": MagicDictionaryPattern,
    "near": MagicDictionaryNear,
    "query": MagicDictionaryQuery,
    "trie": MagicDictionaryTrie,
}

cases = [
    (["hello", "leetcode"], ["hello", "hhllo", "hell", "leetcoded", "hallo", "leetcodf", "xeetcode"]),
    (["abc", "abd", "xyz"], ["abx", "abc", "xyz", "abcd", "xyy", "ab", "ayz", "zzz"]),
    (["a"], ["a", "b", "z", "aa", ""]),                  # một chữ cái (chuỗi rỗng ngoài ràng buộc, chỉ để kiểm tính bền)
    (["a", "b"], ["a", "b", "c"]),                       # các từ đều cách nhau 1 ký tự
    (["aa", "ab", "ba"], ["aa", "ab", "ba", "bb", "ac", "cc"]),
    (["hello", "hallo"], ["hello", "hallo", "hxllo", "hxlla"]),
    (["abcde"], ["abcde", "abcdx", "xbcde", "abxde", "abxdx", "abcd", "abcdef"]),
    (["z" * 100], ["z" * 100, "z" * 99 + "a", "a" + "z" * 99, "a" * 100]),
]
for dictionary, queries in cases:
    ops = ["MagicDictionary", "buildDict"] + ["search"] * len(queries)
    args = [[], [dictionary]] + [[q] for q in queries]
    want = [None, None] + [oracle_search(dictionary, q) for q in queries]
    for name, cls in all_classes.items():
        got = run_commands(cls, ops, args)
        assert got == want, (name, dictionary, queries, got, want)
print("OK:", len(cases), "bộ ca cố định, 5 cách khớp oracle")

OK: 8 bộ ca cố định, 5 cách khớp oracle


### Kết quả mong đợi viết tay (không dùng oracle)

Vài ca cần đúng **theo đề**, ta ghi tay để không phụ thuộc `oracle_search` (phòng khi chính oracle sai).

In [22]:
hand = [
    (["hello", "leetcode"], "hello", False),      # khớp chính xác không tính
    (["hello", "leetcode"], "hhllo", True),
    (["hello", "leetcode"], "hell", False),       # độ dài khác
    (["hello", "leetcode"], "leetcoded", False),
    (["hello", "hallo"], "hello", True),          # hallo cách hello đúng 1 ký tự
    (["abc"], "abc", False),
    (["abc"], "abd", True),
    (["abc"], "xyz", False),                      # khác 3 ký tự
    (["abc"], "xbz", False),                      # khác 2 ký tự
    (["a"], "b", True),
    (["a"], "a", False),
]
for dictionary, w, want in hand:
    for name, cls in all_classes.items():
        obj = cls()
        obj.buildDict(dictionary)
        assert obj.search(w) == want, (name, dictionary, w)
print("OK:", len(hand), "ca viết tay x 5 cách")

OK: 11 ca viết tay x 5 cách


### Test ngẫu nhiên (seed cố định, đúng ràng buộc)

Từ điển 1–8 từ **khác nhau** trên bảng chữ `abc`, độ dài 1–5; truy vấn gồm từ ngẫu nhiên (độ dài 1–6) và từ **sinh ra bằng cách đổi 0 hoặc 1 ký tự** của một từ trong từ điển (để có nhiều ca `True`). So 5 cách với `oracle_search`.

In [23]:
def rand_word(rng, alpha, lo, hi):
    return "".join(rng.choice(alpha) for _ in range(rng.randint(lo, hi)))

def make_case(rng, alpha="abc"):
    k = rng.randint(1, 8)
    words = set()
    while len(words) < k:
        words.add(rand_word(rng, alpha, 1, 5))
    words = sorted(words)
    rng.shuffle(words)
    queries = []
    for _ in range(20):
        if rng.random() < 0.4:
            queries.append(rand_word(rng, alpha, 1, 6))
        else:
            x = rng.choice(words)
            i = rng.randrange(len(x))
            queries.append(x[:i] + rng.choice(alpha) + x[i + 1:])    # có thể trùng x (đổi 0 ký tự)
    return words, queries

rng = random.Random(676)
n_true = n_false = 0
for _ in range(1500):
    dictionary, queries = make_case(rng)
    ops = ["MagicDictionary", "buildDict"] + ["search"] * len(queries)
    args = [[], [dictionary]] + [[q] for q in queries]
    want = [None, None] + [oracle_search(dictionary, q) for q in queries]
    n_true += sum(1 for v in want if v is True)
    n_false += sum(1 for v in want if v is False)
    for name, cls in all_classes.items():
        got = run_commands(cls, ops, args)
        assert got == want, (name, dictionary, queries)
print("OK: 1500 bộ ngẫu nhiên; số kết quả True =", n_true, ", False =", n_false)
assert n_true > 1000 and n_false > 1000

OK: 1500 bộ ngẫu nhiên; số kết quả True = 15271 , False = 14729


### Test cỡ tối đa

Từ điển 100 từ độ dài 100; 100 truy vấn (một nửa là từ điển đổi 1 ký tự, một nửa đổi 2 ký tự hoặc ngẫu nhiên). Dùng bảng chữ rộng (26 chữ) và bảng hẹp (`ab`, nhiều từ gần nhau) để thử cả hai kiểu. Cách 1 làm tham chiếu.

In [24]:
def big_case(rng, alpha):
    words = set()
    while len(words) < 100:
        words.add("".join(rng.choice(alpha) for _ in range(100)))
    words = list(words)
    queries = []
    for t in range(100):
        x = list(rng.choice(words))
        flips = 1 if t % 2 == 0 else rng.choice([0, 2, 3])
        for _ in range(flips):
            i = rng.randrange(100)
            x[i] = rng.choice(alpha)
        queries.append("".join(x))
    return words, queries

rng = random.Random(6760)
for alpha in (ascii_lowercase, "ab"):
    words, queries = big_case(rng, alpha)
    ops = ["MagicDictionary", "buildDict"] + ["search"] * 100
    args = [[], [words]] + [[q] for q in queries]
    ref = run_commands(MagicDictionaryBrute, ops, args)
    for name, cls in all_classes.items():
        assert run_commands(cls, ops, args) == ref, name
    print(f"bảng chữ {len(alpha)} ký tự: True = {sum(1 for v in ref if v is True)}, False = {sum(1 for v in ref if v is False)}")
print("OK: test cỡ tối đa, 5 cách khớp nhau")

bảng chữ 26 ký tự: True = 49, False = 51


bảng chữ 2 ký tự: True = 35, False = 65
OK: test cỡ tối đa, 5 cách khớp nhau


### Test gọi `buildDict` hai lần và nhiều đối tượng độc lập

Đề đảm bảo `buildDict` chỉ gọi một lần, nhưng cài đặt tốt nên **không dính trạng thái cũ** nếu bị gọi lại, và hai đối tượng khác nhau **không ảnh hưởng nhau** (liên quan lỗi 4 ở mục 13).

In [25]:
for name, cls in all_classes.items():
    a, b = cls(), cls()
    a.buildDict(["abc"])
    b.buildDict(["xyz"])
    assert a.search("abd") is True and a.search("xyd") is False, name
    assert b.search("xyd") is True and b.search("abd") is False, name
    a.buildDict(["qqq"])                     # xây lại: từ điển cũ phải biến mất
    assert a.search("abd") is False and a.search("qqa") is True, name
print("OK: đối tượng độc lập và xây lại từ điển đúng")

OK: đối tượng độc lập và xây lại từ điển đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi khớp chính xác là `True` (quên "tự khớp")

Với cách mẫu `*`, nếu chỉ kiểm tra "mẫu có trong bảng" mà không đếm, thì `searchWord` có trong từ điển sẽ khớp chính mẫu của nó và trả `True` dù không có từ nào khác cách nó một ký tự.

In [26]:
class WrongSelfMatch:
    def buildDict(self, dictionary):
        self.pats = set()
        for w in dictionary:
            for i in range(len(w)):
                self.pats.add(w[:i] + "*" + w[i + 1:])

    def search(self, searchWord):
        w = searchWord
        return any(w[:i] + "*" + w[i + 1:] in self.pats for i in range(len(w)))   # SAI: quên đếm

obj = WrongSelfMatch()
obj.buildDict(["hello", "leetcode"])
print("sai    : search('hello') =", obj.search("hello"), "| đúng:", oracle_search(["hello", "leetcode"], "hello"))
assert obj.search("hello") is True
assert oracle_search(["hello", "leetcode"], "hello") is False

sai    : search('hello') = True | đúng: False


**Vì sao sai?** Mẫu `h*llo` của `hello` có sẵn trong bảng vì chính `hello` đã đóng góp nó. **Sửa:** dùng `Counter`/dict đếm và điều kiện `c > 1 or (c == 1 and w not in words)` như cách 2; hoặc dùng cách 3/4 (không bao giờ sinh lại chính `w`).

### Lỗi 2 — So sánh khi độ dài khác nhau (`zip` cắt cụt)

`zip` dừng ở chuỗi ngắn hơn. Nếu không kiểm tra độ dài, `"hxl"` bị coi là "cách `hello` một ký tự" (chỉ so 3 cặp đầu: `h=h`, `x≠e`, `l=l`).

In [27]:
class WrongLength:
    def buildDict(self, dictionary):
        self.words = list(dictionary)

    def search(self, searchWord):
        return any(sum(a != b for a, b in zip(x, searchWord)) == 1 for x in self.words)   # SAI: thiếu len

obj = WrongLength()
obj.buildDict(["hello", "leetcode"])
print("sai    : search('hxl') =", obj.search("hxl"), "| đúng:", oracle_search(["hello", "leetcode"], "hxl"))
assert obj.search("hxl") is True
assert oracle_search(["hello", "leetcode"], "hxl") is False
for name, cls in all_classes.items():
    o = cls(); o.buildDict(["hello", "leetcode"])
    assert o.search("hxl") is False, name
print("5 cách đúng đều trả False")

sai    : search('hxl') = True | đúng: False
5 cách đúng đều trả False


**Sửa:** thêm `len(x) == len(searchWord)` (cách 1); các cách khác tự xử lý nhờ độ dài mẫu/trie.

### Lỗi 3 — Cho phép `diff <= 1` (đổi 0 ký tự cũng tính)

Đề yêu cầu đổi **đúng một** ký tự; "không đổi" (từ có sẵn trong từ điển) **không** đủ.

In [28]:
class WrongAtMostOne:
    def buildDict(self, dictionary):
        self.words = list(dictionary)

    def search(self, searchWord):
        for x in self.words:
            if len(x) == len(searchWord) and sum(a != b for a, b in zip(x, searchWord)) <= 1:   # SAI: <= 1
                return True
        return False

obj = WrongAtMostOne()
obj.buildDict(["hello", "leetcode"])
print("sai    : search('hello') =", obj.search("hello"), "| đúng: False")
assert obj.search("hello") is True
good = MagicDictionaryBrute(); good.buildDict(["hello", "leetcode"])
assert good.search("hello") is False

sai    : search('hello') = True | đúng: False


**Sửa:** điều kiện `diff == 1` (so bằng, không phải `<=`).

### Lỗi 4 — Trạng thái đặt ở mức class

Khai báo `words = set()` ngay trong thân class làm mọi đối tượng **dùng chung** một `set`. LeetCode có thể tạo nhiều đối tượng trong cùng tiến trình, nên từ điển của lần chạy trước còn dính ở lần sau.

In [29]:
class WrongShared:
    words = set()                          # SAI: biến mức class, dùng chung

    def buildDict(self, dictionary):
        self.words.update(dictionary)      # thay đổi chính tập dùng chung

    def search(self, searchWord):
        w = searchWord
        return any(w[:i] + c + w[i + 1:] in self.words
                   for i in range(len(w)) for c in ascii_lowercase if c != w[i])

a = WrongShared(); a.buildDict(["abc"])
b = WrongShared(); b.buildDict(["xyz"])
print("b.words =", b.words, "<- còn từ của a")
print("b.search('abd') =", b.search("abd"), "(đúng phải là False)")
assert b.search("abd") is True

b.words = {'xyz', 'abc'} <- còn từ của a
b.search('abd') = True (đúng phải là False)


**Sửa:** gán trong `__init__`: `self.words = set()` — mỗi đối tượng có tập riêng (như mọi cách giải của notebook này).

### Lỗi 5 — Trie DFS quên kiểm tra "kết thúc từ"

Nếu khi `i == n` chỉ kiểm tra `diff == 1` mà không kiểm tra nút có đánh dấu kết thúc từ, thì một **tiền tố** của từ trong từ điển bị coi là từ: `"hxl"` đi theo `h-e-l` (đổi `x` thành `e`) tới hết chuỗi với `diff = 1` và trả `True`, dù `hel` không phải từ trong từ điển.

In [30]:
class WrongTrieNoEnd:
    def buildDict(self, dictionary):
        self.root = {}
        for w in dictionary:
            node = self.root
            for ch in w:
                node = node.setdefault(ch, {})
            node["#"] = True

    def search(self, searchWord):
        w, n = searchWord, len(searchWord)

        def dfs(node, i, diff):
            if i == n:
                return diff == 1                    # SAI: thiếu  and "#" in node
            for ch, child in node.items():
                if ch == "#":
                    continue
                d = diff + (ch != w[i])
                if d <= 1 and dfs(child, i + 1, d):
                    return True
            return False

        return dfs(self.root, 0, 0)

obj = WrongTrieNoEnd()
obj.buildDict(["hello", "leetcode"])
print("sai    : search('hxl') =", obj.search("hxl"), "| đúng: False")
assert obj.search("hxl") is True
ok = MagicDictionaryTrie(); ok.buildDict(["hello", "leetcode"])
assert ok.search("hxl") is False

sai    : search('hxl') = True | đúng: False


**Sửa:** `return diff == 1 and "#" in node` — chỉ thành công khi đường đi trong trie kết thúc **đúng** tại một từ đã lưu.

## 14. Mẹo & nhận dạng pattern

- **"Đổi đúng một ký tự / khác đúng một vị trí"** → nghĩ ngay tới **mẫu có `*`** (wildcard / masking) hoặc **liệt kê hàng xóm**. Cùng họ với *Word Ladder* (127): các từ cách nhau một ký tự ↔ cùng mẫu `h*llo`.
- Khi phép so sánh cần "**khác 0**" (không tự khớp), nghĩ về cách loại **chính nó**: `c ≠ ch` khi sinh hàng xóm, hoặc đếm và trừ đi chính nó.
- **Tập hợp / bảng băm** biến "quét danh sách" thành "tra O(1)" — đổi lại bộ nhớ.
- **Trie** hợp khi cần tìm theo **tiền tố** hoặc "gần đúng với sai số nhỏ" (DFS mang theo số lần sai). Mẫu `dfs(node, i, diff)` tái dùng cho bài *Search Word – Data Structure* (211, có `.`) và các biến thể sai `k` ký tự.
- **Bài thiết kế lớp:** trạng thái nằm trong `__init__` (`self.x`), không để ở mức class; đọc kỹ giao diện (`void` trả `None`); viết `run_commands` để tự mô phỏng chuỗi lệnh.
- Chọn cách theo bài: `N` lớn, truy vấn ít → cách 4 hoặc trie (nhẹ bộ nhớ); truy vấn nhiều → cách 2/3.
- Kiểm tra nhanh các ca: từ có sẵn (phải `False` nếu không có hàng xóm), từ khác độ dài, từ khác 2 ký tự, từ điển một chữ cái.

## 15. Biến thể

### 15.1. Đổi **đúng `k`** ký tự (tổng quát hóa trie + DFS)

Chỉ cần thay `1` thành `k` trong điều kiện: cho phép `diff` tới `k`, thành công khi `i == n`, `diff == k` và nút kết thúc từ.

In [31]:
class MagicDictionaryK:
    def __init__(self):
        self.root = {}

    def buildDict(self, dictionary):
        self.root = {}
        for w in dictionary:
            node = self.root
            for ch in w:
                node = node.setdefault(ch, {})
            node["#"] = True

    def search(self, searchWord, k):
        w, n = searchWord, len(searchWord)

        def dfs(node, i, diff):
            if i == n:
                return diff == k and "#" in node
            for ch, child in node.items():
                if ch == "#":
                    continue
                d = diff + (ch != w[i])
                if d <= k and dfs(child, i + 1, d):
                    return True
            return False

        return dfs(self.root, 0, 0)

m = MagicDictionaryK()
m.buildDict(["hello", "leetcode"])
print([m.search("hello", k) for k in range(4)])          # [True, False, False, False]
print([m.search("hxllx", k) for k in range(4)])          # [False, False, True, False]
assert m.search("hello", 0) is True and m.search("hello", 1) is False
assert m.search("hxllx", 2) is True and m.search("hxllx", 1) is False

# so với vét cạn
rng = random.Random(15)
for _ in range(300):
    words, queries = make_case(rng)
    mk = MagicDictionaryK(); mk.buildDict(words)
    for q in queries:
        for k in range(0, 4):
            want = any(len(x) == len(q) and sum(a != b for a, b in zip(x, q)) == k for x in words)
            assert mk.search(q, k) == want
print("MagicDictionaryK khớp vét cạn")

[True, False, False, False]
[False, False, True, False]
MagicDictionaryK khớp vét cạn


### 15.2. Cách nhau **một phép chỉnh sửa** (thêm / xóa / đổi một ký tự)

Mở rộng "khoảng cách chỉnh sửa bằng 1". Hai chuỗi cách nhau đúng một phép chỉnh sửa nếu: cùng độ dài và khác đúng một vị trí; hoặc độ dài hơn kém 1 và xóa một ký tự của chuỗi dài được chuỗi ngắn. Hàm kiểm tra (dùng vét cạn trên từ điển):

In [32]:
def edit_distance_one(a, b):
    if a == b:
        return False
    if len(a) == len(b):
        return sum(x != y for x, y in zip(a, b)) == 1
    if len(a) > len(b):
        a, b = b, a                        # a ngắn hơn
    if len(b) - len(a) != 1:
        return False
    return any(b[:i] + b[i + 1:] == a for i in range(len(b)))

def search_edit_one(dictionary, w):
    return any(edit_distance_one(x, w) for x in dictionary)

print(search_edit_one(["hello", "leetcode"], "hell"))       # True: xóa 'o'
print(search_edit_one(["hello", "leetcode"], "helloo"))     # True: thêm 'o'
print(search_edit_one(["hello", "leetcode"], "hello"))      # False
assert search_edit_one(["hello"], "hell") is True
assert search_edit_one(["hello"], "hel") is False

True
True
False


### 15.3. Đếm số từ trong từ điển cách `searchWord` đúng một ký tự

Mẹo từ cách 2: một từ `x` cách `w` đúng một vị trí **chỉ khớp một** mẫu của `w` (mẫu ở vị trí khác biệt đó). Vậy số từ = `Σ_i cnt[mẫu_i(w)]` trừ đi phần "tự khớp": nếu `w` có trong từ điển thì nó đã bị đếm `L` lần (một lần mỗi mẫu) → trừ `L`.

In [33]:
def count_one_away(dictionary, w):
    cnt = {}
    for x in dictionary:
        for i in range(len(x)):
            p = x[:i] + "*" + x[i + 1:]
            cnt[p] = cnt.get(p, 0) + 1
    total = sum(cnt.get(w[:i] + "*" + w[i + 1:], 0) for i in range(len(w)))
    if w in set(dictionary):
        total -= len(w)
    return total

d = ["hello", "hallo", "hullo", "leetcode"]
print(count_one_away(d, "hello"), count_one_away(d, "hxllo"), count_one_away(d, "leetcode"))
assert count_one_away(d, "hello") == 2          # hallo, hullo
assert count_one_away(d, "hxllo") == 3
assert count_one_away(d, "leetcode") == 0

rng = random.Random(153)
for _ in range(300):
    words, queries = make_case(rng)
    for q in queries:
        want = sum(1 for x in words if len(x) == len(q) and sum(a != b for a, b in zip(x, q)) == 1)
        assert count_one_away(words, q) == want
print("count_one_away khớp vét cạn")

2 3 0


count_one_away khớp vét cạn


### 15.4. Bài liên quan

- **208. Implement Trie (Prefix Tree)** — nền tảng trie.
- **211. Design Add and Search Words Data Structure** — trie + DFS với ký tự `.`.
- **127. Word Ladder** — hai từ liền kề khi cách nhau đúng một ký tự; dùng mẫu `*` để dựng đồ thị.
- **677. Map Sum Pairs** — trie lưu giá trị.

## 16. So sánh các cách

| Cách | Ý tưởng | `buildDict` | `search` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. So từng từ | đếm vị trí khác với mọi từ cùng độ dài | O(N) | O(N·L) | O(N·L) | N nhỏ, cần code nhanh, làm oracle |
| 2. Mẫu `*` + Counter | cùng mẫu ⇔ khác đúng một vị trí; đếm để loại tự khớp | O(N·L²) | O(L²) | O(N·L²) | **mặc định nên dùng**; nhiều truy vấn |
| 3. Tập hàng xóm | tạo sẵn mọi chuỗi khác một ký tự | O(25·N·L²) | O(L) | O(25·N·L²) | truy vấn cực nhiều, RAM thoải mái |
| 4. Thử thay ký tự | sinh hàng xóm của `searchWord`, tra tập | O(N·L) | O(25·L²) | O(N·L) | tiết kiệm bộ nhớ, code ngắn |
| 5. Trie + DFS | duyệt trie song song với `searchWord`, đếm `diff` | O(N·L) | O(26·L²) tệ nhất | O(N·L) | từ chung tiền tố; mở rộng `k` ký tự |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell **TODO** chỉ gán `None` hoặc định nghĩa hàm/lớp rỗng — **không được gọi** trước khi bạn viết xong. Đáp án ở mục 19.

### Bài 1 (đoán output)

Không chạy code, hãy đoán `result` (một list các chuỗi) rồi điền vào `my_guess_1`.

In [34]:
w = "cat"
result = [w[:i] + "*" + w[i + 1:] for i in range(len(w))]
# print(result)   # <- bỏ dấu # sau khi đã đoán xong để kiểm tra

my_guess_1 = None    # TODO: điền list bạn đoán

### Bài 2 (tính tay)

Từ điển `["abc", "abd", "xyz"]`. Tính tay kết quả của các lần `search` theo thứ tự `["abx", "abc", "xyz", "abcd", "xyy"]` (ví dụ 2 ở mục 2 đã cho đáp án — hãy **tự suy luận** từng cái trước, đừng nhìn). Điền list `True`/`False` vào `my_answers_2`.

In [35]:
my_answers_2 = None    # TODO: điền list 5 giá trị True/False

### Bài 3 (điền TODO)

Hoàn thành lớp (cách 4: thử thay ký tự của `searchWord` rồi tra tập): thay các `...` bằng biểu thức/câu lệnh đúng.

In [36]:
class Exercise3:
    def __init__(self):
        self.words = set()

    def buildDict(self, dictionary):
        self.words = ...                       # TODO: lưu từ điển vào một set

    def search(self, searchWord):
        w = searchWord
        for i, ch in enumerate(w):
            for c in ascii_lowercase:
                if ...:                        # TODO: ký tự thay thế phải KHÁC ch
                    cand = ...                 # TODO: chuỗi sau khi đổi ký tự thứ i thành c
                    if cand in self.words:
                        return True
        return False

### Bài 4 (tự viết lại)

Tự viết lớp `Exercise4` có `buildDict`, `search` theo **cách 1 (so từng từ)** — nhớ kiểm tra độ dài và dùng `== 1`. Không nhìn lại mục 5.

In [37]:
class Exercise4:
    def buildDict(self, dictionary):
        # TODO
        pass

    def search(self, searchWord):
        # TODO
        pass

### Bài 5 (biến thể / mini challenge)

Viết hàm `exercise5(dictionary, word)` trả về **list các từ trong từ điển cách `word` đúng một ký tự**, **sắp xếp tăng dần**. Ví dụ `exercise5(["hello", "hallo", "hullo", "leetcode"], "hello")` → `["hallo", "hullo"]`; `exercise5(["abc"], "abc")` → `[]`.

In [38]:
def exercise5(dictionary, word):
    # TODO: viết lời giải
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `range(len(w))` cho `i = 0, 1, 2`.
- Mức 2: với mỗi `i`, `w[:i]` là phần trước, `w[i+1:]` là phần sau, giữa chèn `"*"`.
- Mức 3: `i = 0` → `"" + "*" + "at"`; viết nốt hai mẫu còn lại.

**Bài 2**
- Mức 1: với mỗi truy vấn, tìm từ **cùng độ dài** trong từ điển.
- Mức 2: đếm số vị trí khác nhau với từng từ đó; cần đúng `1`.
- Mức 3: `abx` khác `abc` (1 vị trí) → True; `abc` khác `abd` (1 vị trí) → True; `xyz` khác `abc`, `abd` ở 3 và 3 vị trí → False; `abcd` độ dài 4 → False; `xyy` khác `xyz` (1) → True.

**Bài 3**
- Mức 1: nhìn lại code cách 4 (mục 7.3).
- Mức 2: `buildDict` chỉ cần `set(dictionary)`; điều kiện loại ký tự giống ban đầu là `c != ch`.
- Mức 3: `self.words = set(dictionary)`; `if c != ch:`; `cand = w[:i] + c + w[i + 1:]`.

**Bài 4**
- Mức 1: `buildDict` lưu danh sách.
- Mức 2: `search` duyệt từng từ; bỏ qua nếu độ dài khác.
- Mức 3: đếm `sum(a != b for a, b in zip(x, searchWord))`; trả `True` nếu bằng `1`.

**Bài 5**
- Mức 1: giống cách 1 nhưng **thu thập** thay vì trả `True` ngay.
- Mức 2: dùng list comprehension với điều kiện độ dài bằng nhau và đếm khác nhau bằng `1`.
- Mức 3: `sorted(x for x in dictionary if len(x) == len(word) and sum(a != b for a, b in zip(x, word)) == 1)`.

## 19. Đáp án tham khảo

Chạy các cell sau để kiểm tra bằng `assert`.

In [39]:
# Bài 1
w = "cat"
answer_1 = [w[:i] + "*" + w[i + 1:] for i in range(len(w))]
print("Bài 1:", answer_1)
assert answer_1 == ["*at", "c*t", "ca*"]

# Bài 2
dictionary2 = ["abc", "abd", "xyz"]
answer_2 = [oracle_search(dictionary2, q) for q in ["abx", "abc", "xyz", "abcd", "xyy"]]
print("Bài 2:", answer_2)
assert answer_2 == [True, True, False, False, True]

Bài 1: ['*at', 'c*t', 'ca*']
Bài 2: [True, True, False, False, True]


In [40]:
# Bài 3
class Exercise3Solution:
    def __init__(self):
        self.words = set()

    def buildDict(self, dictionary):
        self.words = set(dictionary)

    def search(self, searchWord):
        w = searchWord
        for i, ch in enumerate(w):
            for c in ascii_lowercase:
                if c != ch:
                    cand = w[:i] + c + w[i + 1:]
                    if cand in self.words:
                        return True
        return False

# Bài 4
class Exercise4Solution:
    def buildDict(self, dictionary):
        self.words = list(dictionary)

    def search(self, searchWord):
        for x in self.words:
            if len(x) == len(searchWord) and sum(a != b for a, b in zip(x, searchWord)) == 1:
                return True
        return False

# Bài 5
def exercise5_solution(dictionary, word):
    return sorted(x for x in dictionary
                  if len(x) == len(word) and sum(a != b for a, b in zip(x, word)) == 1)

for ops, args, exp in examples:
    assert run_commands(Exercise3Solution, ops, args) == exp
    assert run_commands(Exercise4Solution, ops, args) == exp
assert exercise5_solution(["hello", "hallo", "hullo", "leetcode"], "hello") == ["hallo", "hullo"]
assert exercise5_solution(["abc"], "abc") == []
print("Đáp án bài 3, 4, 5 đúng trên ví dụ")

rng = random.Random(19)
for _ in range(500):
    words, queries = make_case(rng)
    a, b = Exercise3Solution(), Exercise4Solution()
    a.buildDict(words); b.buildDict(words)
    for q in queries:
        want = oracle_search(words, q)
        assert a.search(q) == want and b.search(q) == want
        assert (len(exercise5_solution(words, q)) > 0) == want
print("Đối chiếu ngẫu nhiên OK")

Đáp án bài 3, 4, 5 đúng trên ví dụ


Đối chiếu ngẫu nhiên OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (mẫu `*` + đếm; truy vấn O(L²), không phụ thuộc số từ):**

```python
class MagicDictionary:

    def __init__(self):
        self.words = set()
        self.cnt = {}

    def buildDict(self, dictionary: List[str]) -> None:
        self.words = set(dictionary)
        self.cnt = {}
        for w in dictionary:
            for i in range(len(w)):
                p = w[:i] + "*" + w[i + 1:]          # che vị trí i
                self.cnt[p] = self.cnt.get(p, 0) + 1

    def search(self, searchWord: str) -> bool:
        w = searchWord
        for i in range(len(w)):
            p = w[:i] + "*" + w[i + 1:]
            c = self.cnt.get(p, 0)
            if c > 1 or (c == 1 and w not in self.words):   # có từ KHÁC w cùng mẫu
                return True
        return False
```

**Checklist trước khi submit:**

- [ ] Đổi **đúng một** ký tự: khớp chính xác (0 khác biệt) **không** được tính.
- [ ] Từ khác độ dài không bao giờ khớp (mẫu / trie / `len` tự lo; nếu dùng `zip` phải tự kiểm tra `len`).
- [ ] Nếu dùng mẫu: **đếm** số từ theo mẫu, và điều kiện `c > 1 or (c == 1 and w not in words)`.
- [ ] Nếu dùng trie: khi hết chuỗi cần `diff == 1` **và** nút là kết thúc từ.
- [ ] Trạng thái nằm trong `__init__` (`self.*`), không để biến mutable ở mức class.
- [ ] `buildDict` reset trạng thái (không cộng dồn lần gọi trước).
- [ ] Đã thử: từ có sẵn trong từ điển, từ khác 2 ký tự, khác độ dài, từ điển một chữ, hai từ cách nhau một ký tự.
- [ ] Đã so với `oracle_search` bằng test ngẫu nhiên có seed; độ phức tạp `search` ≈ O(L²) (cách nộp).